# Pilot 2025 Aroma Model Selection and MBDoE

This notebook documents the model-structure iteration used to improve the aroma layer for the pilot-scale natural-must dataset. The goal is not only to reduce error, but to choose a structure that is interpretable, numerically identifiable, and useful for future model-based experimental design.

## Workflow

1. Load the curated pilot dataset, including the `25171` effective start correction at `t_original = 100 h`.
2. Keep the calibrated primary fermentation, glycerol, and secondary v2 model as the current prior.
3. Fit several aroma kinetic variants that differ only in the ethyl-acetate production structure.
4. Compare variants using weighted residual error, AICc/BIC, active-bound diagnostics, and ethyl-acetate retained-pool bias.
5. Recompute the local Fisher information matrix for the selected structure.
6. Evaluate candidate natural-must designs using FIM accumulation and select an optimal campaign with a hybrid criterion.

## Observation Model

For each aroma species `i`, the workbook contains total aroma and condenser-equivalent aroma:

$$C_{total,i}^{obs}=C_{wine,i}^{obs}+C_{cond,i}^{obs}$$

so the retained liquid concentration fitted by the ODE is

$$C_{wine,i}^{obs}=C_{total,i}^{obs}-C_{cond,i}^{obs}.$$

This reconstruction is applied only when the condenser-equivalent value is positive. Exact zero entries in the condenser columns are treated as missing or below-reporting-limit records, because the condenser behaves as an accumulated trap rather than a continuously sampled online analyzer. When total aroma is available but positive condenser information is not, the fitted observation is `C_total = C_L + C_cond`.

The model states are

$$x_{a,i}=\left[C_{L,i}, C_{cond,i}\right]^T$$

with dynamics

$$\frac{dC_{L,i}}{dt}=r_{prod,i}-r_{loss,i}, \qquad \frac{dC_{cond,i}}{dt}=r_{loss,i}.$$

Volatilization is kept common across variants and follows the CO2-stripping balance used in the Mouret-style aroma partition framework:

$$r_{loss,i}=\alpha_i K_i(T,E)q_{CO2}C_{L,i}.$$

In this run, `K_i` is not fitted. It is computed as

$$K_i(T,E)=\frac{\gamma_i^{UNIFAC}(T,x_{water},x_{ethanol})P_i^{sat,Antoine}(T)}{RT C_{tot,L}}$$

where `gamma_i` is the infinite-dilution activity coefficient from original UNIFAC for the water-ethanol mixture, `P_i^{sat}` is calculated from Antoine coefficients, and `C_{tot,L}` is the total liquid molar concentration. `alpha_i` remains an effective capture/loss scaling parameter estimated from the condenser data. The audit tables `partition_antoine_coefficients.csv` and `partition_antoine_unifac_audit.csv` document the thermodynamic inputs.

## Candidate Kinetic Structures

### `baseline_phase`

Original phase-dependent aroma model used in the previous pilot calibration.

$$r_{EA}=\left(k_{EA,g}\phi_N+k_{EA,s}(1-\phi_N)\right)q_S$$

Reference empirical model: aroma synthesis is tied to sugar uptake and split between nitrogen-associated growth and stationary phases; volatilization is CO2-rate dependent.

### `ea_biomass_background`

Tests whether ethyl acetate behaves as a biomass/activity output rather than a direct sugar-rate output.

$$r_{EA}=r_{EA,phase}+k_{EA,X}X$$

Adds a biomass-associated ethyl-acetate formation term, representing acetyltransferase capacity and natural-must precursor availability not captured by sugar uptake alone.

### `ea_ethanol_biomass`

Tests whether the missing ethyl-acetate source is ethanol/biomass dependent.

$$r_{EA}=r_{EA,phase}+k_{EA,XE}X\frac{E}{K_E+E}$$

Ethyl acetate formation requires ethanol as the alcohol precursor. This variant adds a saturating ethanol-biomass term while keeping volatilization unchanged.

### `ea_ethanol_temperature`

Tests whether the missing ethyl-acetate source is mainly temperature-modulated.

$$r_{EA}=r_{EA,phase}+k_{EA,XE}X\frac{E}{K_E+E}q_{10,EA}^{(T-20)/10}$$

Fermentative ester synthesis and stripping are temperature-sensitive. This variant lets the ethanol-biomass production term follow a fitted Q10 factor around 20 C.

### `ea_ethanol_nlimited`

Tests whether ethyl acetate increases when ethanol is available and nitrogen is depleted.

$$r_{EA}=r_{EA,phase}+k_{EA,XE}X\frac{E}{K_E+E}+k_{EA,XE,Nlim}X\frac{E}{K_E+E}\frac{K_N}{K_N+N}$$

Nitrogen limitation changes yeast aroma metabolism. This variant separates a baseline ethanol-biomass source from an additional nitrogen-limited source.

### `ea_redox_acetaldehyde`

Tests whether secondary redox chemistry explains the ethyl-acetate deficit.

$$r_{EA}=r_{EA,phase}+k_{EA,XE}X\frac{E}{K_E+E}+k_{EA,AcAld}X\frac{AcAld}{K_{AcAld}+AcAld}\frac{E}{K_E+E}$$

Ethyl acetate is linked to acetyl-CoA/redox and acetaldehyde/acetate-side metabolism. The reduced secondary model provides acetaldehyde as a proxy state.

### `ea_combined_parsimonious`

Upper-complexity check for whether several mechanistic proxies are simultaneously needed.

$$r_{EA}=r_{EA,phase}+k_{EA,X}X+k_{EA,XE}X\frac{E}{K_E+E}+k_{EA,AcAld}X\frac{AcAld}{K_{AcAld}+AcAld}\frac{E}{K_E+E}$$

A compact combined model: background biomass capacity, ethanol precursor availability, and acetaldehyde/redox proxy. This is the largest variant considered here and is penalized by BIC/AICc.

## Statistical Criteria

Let `r(theta)` be the weighted residual vector. The data-only weighted sum of squared errors is

$$WSSE= r(\theta)^T r(\theta).$$

For model comparison, the notebook reports

$$AIC_c = WSSE + 2k + \frac{2k(k+1)}{n-k-1},$$

$$BIC = WSSE + k\log(n),$$

where `k` is the number of fitted aroma parameters and `n` is the number of aroma residuals. Because ethyl acetate is the priority output and the previous structure showed systematic underprediction, ethyl-acetate residuals are weighted twice as strongly as the other aroma residuals in the fitting objective. The final selection score is BIC plus penalties for active bounds and unacceptable ethyl-acetate retained-pool bias/RMSE. This prevents selection of a flexible model that only improves the global objective while still failing the main problematic output.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
cwd = Path.cwd()
if (cwd / 'results' / 'aroma_model_selection_doe').exists():
    RESULTS = cwd / 'results' / 'aroma_model_selection_doe'
else:
    RESULTS = cwd / 'fermentation_model' / 'pilot_2025' / 'results' / 'aroma_model_selection_doe'
model_summary = pd.read_csv(RESULTS / 'model_selection_summary.csv')
variant_metrics = pd.read_csv(RESULTS / 'variant_pool_metrics.csv')
theta = pd.read_csv(RESULTS / 'theta_selected_aroma_model.csv', index_col=0)
estimability = pd.read_csv(RESULTS / 'parameter_estimability_selected_current.csv')
weak = pd.read_csv(RESULTS / 'weak_directions_selected_current.csv')
ranking = pd.read_csv(RESULTS / 'candidate_ranking_selected_model.csv')
campaign = pd.read_csv(RESULTS / 'selected_campaign_hybrid_selected_model.csv')
model_summary.sort_values('selection_score')

## Selected Parameter Vector

In [ ]:
theta

## Pool-Level Fit Metrics

In [ ]:
variant_metrics.sort_values(['model', 'species', 'pool']).head(80)

## Model-Comparison Plot

In [ ]:
plot = RESULTS / 'plots' / 'model_selection_comparison.png'
display(Image(filename=str(plot)))

## Selected Structure: `ea_ethanol_nlimited`

Tests whether ethyl acetate increases when ethanol is available and nitrogen is depleted.

$$r_{EA}=r_{EA,phase}+k_{EA,XE}X\frac{E}{K_E+E}+k_{EA,XE,Nlim}X\frac{E}{K_E+E}\frac{K_N}{K_N+N}$$

Interpretation: this is the simplest structure that best balances ethyl-acetate fit, global aroma error, parameter parsimony, and numerical diagnostics.

## Fisher Information Matrix

The local sensitivity matrix is computed by centered finite differences in log-parameter space:

$$J_{m,j}=\frac{r_m(\theta_j e^{h})-r_m(\theta_j e^{-h})}{2h}.$$

The Fisher information matrix is then

$$F = J^T J.$$

Because perturbations are performed in log-parameter space, the covariance approximation is also in log-parameter units. The reported `approx_95_multiplier` is `exp(1.96 sigma_log)`.

In [ ]:
estimability.sort_values('std_log_approx', ascending=False)

## Weak Eigen-Directions

In [ ]:
weak

## MBDoE Criterion

Candidate experiments are evaluated by adding their expected FIM to the current-data FIM:

$$F_{combined}=F_{current}+\sum_{e \in \mathcal{E}}F_e.$$

The ranking reports D-optimality through `logdet(F)`, E-like robustness through the minimum relative eigenvalue, and A-like behavior through `trace(inv(F))`. The selected campaign uses the hybrid score

$$\Phi_{hybrid}=\log\det(F)+2\log(\lambda_{min}/\lambda_{max})-0.05\log(\mathrm{trace}(F^{-1})).$$

This keeps the D-optimal volume objective but discourages designs that leave a nearly unobservable direction.

In [ ]:
cols = ['candidate', 'family', 'combined_logdet', 'combined_min_relative_eigenvalue', 'aroma_mean_var_reduction', 'aroma_worst_var_reduction', 'N_pulses_kg_m3', 'rationale']
ranking[cols].head(12)

## Selected Campaign

In [ ]:
campaign[['campaign_order', 'candidate', 'family', 'campaign_logdet', 'campaign_min_relative_eigenvalue', 'aroma_mean_var_reduction', 'aroma_worst_var_reduction', 'N_pulses_kg_m3', 'rationale']]

## Input Profiles for Selected Designs

In [ ]:
for path in sorted((RESULTS / 'plots' / 'designs').glob('candidate_inputs_*.png')):
    display(Image(filename=str(path)))

## Selected Model Aroma Fits

In [ ]:
for path in sorted((RESULTS / 'plots' / 'selected_fit').glob('selected_aroma_fit_*.png')):
    display(Image(filename=str(path)))